Kaggle API setup in Colab

In [ ]:
!pip install kaggle

Upload Kaggle API key

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Lip reading dataset
!kaggle datasets download gpiosenka/lip-reading

# Facial emotion dataset
!kaggle datasets download msambare/fer2013

# GRID speech dataset
!kaggle datasets download stanfordu/gridcorpus

In [ ]:
!kaggle datasets download msambare/fer2013
!kaggle datasets download omkargurav/lipreading-dataset
!kaggle datasets download ejlok1/cremad

In [ ]:
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/datasets

In [ ]:
#LIP READING EXPRESSION
!unzip /content/drive/MyDrive/datasets/lip-reading-dataset.zip -d /content/lipreading

In [ ]:
!unzip fer2013.zip #FACIAL EMOTION RECOGNITION

In [ ]:
!unzip cremad.zip #VOICE EMOTION RECOGNITION

In [ ]:
!ls

In [ ]:
!ls train


In [ ]:
import tensorflow as tf

IMG_SIZE = 48
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    color_mode="grayscale"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "test",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    color_mode="grayscale"
)

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(48,48,1)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(7,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(48,48,1)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(7,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

Confusion matrix

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
!ls lipreading

In [ ]:
!ls -R lipreading | head -n 50

Lip Dataset Loader

In [ ]:
lip_ds = tf.keras.utils.image_dataset_from_directory(
    "lipreading/outputs",
    image_size=(64,64),
    batch_size=32
)

Lip reading CNN Model

In [ ]:
lip_model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(64,64,3)),

    tf.keras.layers.Conv2D(32,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256,activation='relu'),
    tf.keras.layers.Dense(50,activation='softmax')
])

In [ ]:
#compile
lip_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lip_model.summary()

Train lip-reading model

In [ ]:
!ls lipreading/outputs/a_1 | head

In [ ]:
import cv2
import numpy as np

X = []
y = []

img_size = 64

for label, cls in enumerate(classes):
    cls_path = os.path.join(base_path, cls)

    for root, dirs, files in os.walk(cls_path):
        for file in files:
            if file.endswith(".jpg") or file.endswith(".png"):
                img_path = os.path.join(root, file)

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (img_size, img_size))

                X.append(img)
                y.append(label)

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

In [ ]:
X = X.reshape(-1,64,64,1) / 255.0

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
import tensorflow as tf

lip_model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(64,64,1)),

    tf.keras.layers.Conv2D(32,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256,activation='relu'),
    tf.keras.layers.Dense(len(classes),activation='softmax')
])

In [ ]:
#compile
lip_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lip_model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=10,
    batch_size=32
)

In [ ]:
print(X_train.shape)
print(y_train.shape)

Lip model evaluate

In [ ]:
loss, acc = lip_model.evaluate(X_test, y_test)
print("Lip model accuracy:", acc)

Prediction Generate

In [ ]:
import numpy as np

y_pred = lip_model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

Confusion Matrix Lip Model

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

Heatmap Visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
sns.heatmap(cm[:20,:20], cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Lip Reading Confusion Matrix (first 20 classes)")
plt.show()

In [ ]:
model.build((None,48,48,1))
lip_model.build((None,64,64,1))

Feature Extractor Model

In [ ]:
from tensorflow.keras.models import Model

emotion_feature_model = Model(
    inputs=model.inputs,
    outputs=model.layers[-2].output
)

lip_feature_model = Model(
    inputs=lip_model.inputs,
    outputs=lip_model.layers[-2].output
)

In [ ]:
print(emotion_feature_model.summary())
print(lip_feature_model.summary())

Extract Features

In [ ]:
#emotion Feature
emotion_features = emotion_feature_model.predict(test_ds)

In [ ]:
#Lip Feature
lip_features = lip_feature_model.predict(X_test)

In [ ]:
min_samples = min(len(emotion_features), len(lip_features))

emotion_features = emotion_features[:min_samples]
lip_features = lip_features[:min_samples]

print(emotion_features.shape)
print(lip_features.shape)

In [ ]:
#Feature fusion
import numpy as np

combined_features = np.concatenate(
    (emotion_features, lip_features),
    axis=1
)

print(combined_features.shape)

In [ ]:
#Final classifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

fusion_model = Sequential([
    Dense(128, activation='relu', input_shape=(384,)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

fusion_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Graphs


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

sns.heatmap(cm[:20,:20],
            cmap="Blues",
            annot=False)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (First 20 Classes)")

plt.show()

In [ ]:
history = model.fit(...)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')

plt.legend(['Train','Validation'])

plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')

plt.legend(['Train','Validation'])

plt.show()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread(list(uploaded.keys())[0])
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.axis("off")

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

faces = face_cascade.detectMultiScale(img, 1.3, 5)

print("Faces detected:", len(faces))

In [ ]:
for (x,y,w,h) in faces:
    cv2.rectangle(img_rgb,(x,y),(x+w,y+h),(255,0,0),2)

plt.imshow(img_rgb)
plt.axis("off")

In [ ]:
for (x,y,w,h) in faces:
    # Assuming mouth is in the lower half of the face
    mouth_y = y + int(h * 2/3)
    mouth_h = h - int(h * 2/3)
    mouth = img[mouth_y : mouth_y + mouth_h, x : x+w]

    mouth_gray = cv2.cvtColor(mouth, cv2.COLOR_BGR2GRAY)
    mouth_gray = cv2.resize(mouth_gray,(64,64))
    mouth_gray = mouth_gray.reshape(1,64,64,1)/255.0

    print(mouth_gray.shape)
    # For simplicity, we process only the first detected mouth.
    # If multiple faces are detected, this will only take the last one in the loop.
    break # Exit loop after processing the first face's mouth

In [ ]:
pred = lip_model.predict(mouth_gray)

label = np.argmax(pred)

print("Predicted lip class:", label)

**Running Demo Model**

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread(list(uploaded.keys())[0])
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.axis("off")

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

faces = face_cascade.detectMultiScale(img,1.3,5)

for (x,y,w,h) in faces:
    cv2.rectangle(img_rgb,(x,y),(x+w,y+h),(255,0,0),2)

plt.imshow(img_rgb)
plt.axis("off")

In [ ]:
for (x,y,w,h) in faces:

    # lips area calculation
    lip_x = x + int(w*0.2)
    lip_y = y + int(h*0.65)

    lip_w = int(w*0.6)
    lip_h = int(h*0.25)

    # draw GREEN rectangle
    cv2.rectangle(
        img_rgb,
        (lip_x,lip_y),
        (lip_x+lip_w,lip_y+lip_h),
        (0,255,0),   # green color
        2
    )

plt.imshow(img_rgb)
plt.axis("off")

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

faces = face_cascade.detectMultiScale(img,1.3,5)

print("Faces detected:",len(faces))

In [ ]:
for (x,y,w,h) in faces:

    face = img[y:y+h, x:x+w]

    mouth = face[int(h*0.6):h,:]

plt.imshow(cv2.cvtColor(mouth, cv2.COLOR_BGR2RGB))
plt.axis("off")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

emotion_labels = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]

# Re-define the emotion detection model
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(48,48,1)),
    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(7,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

for (x,y,w,h) in faces:

    face = img[y:y+h, x:x+w]

    face_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
    face_gray = cv2.resize(face_gray,(48,48))
    face_gray = face_gray.reshape(1,48,48,1)/255.0

emotion_pred = model.predict(face_gray)

emotion_class = np.argmax(emotion_pred)

print("Detected Emotion:", emotion_labels[emotion_class])

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import os

# Define base_path and classes if they are not already in scope
# This assumes 'lipreading/outputs' exists and contains class directories
base_path = "lipreading/outputs"
# Get class names from directory structure to correctly define the model's output layer
classes = sorted(os.listdir(base_path))

# Redefine and compile lip_model if it's not defined, to resolve NameError
# Note: This model will be untrained unless previous training cells are run.
if 'lip_model' not in locals() and 'lip_model' not in globals():
    lip_model = tf.keras.Sequential([
        tf.keras.layers.Rescaling(1./255, input_shape=(64,64,1)),

        tf.keras.layers.Conv2D(32,(3,3),activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256,activation='relu'),
        tf.keras.layers.Dense(len(classes),activation='softmax')
    ])

    lip_model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    print("lip_model was re-defined and compiled as it was not found in scope.")
    print("Warning: This re-defined model is untrained. Please run cells defining and training lip_model for accurate predictions.")

mouth_gray = cv2.cvtColor(mouth, cv2.COLOR_BGR2GRAY)
mouth_gray = cv2.resize(mouth_gray,(64,64))
mouth_gray = mouth_gray.reshape(1,64,64,1)/255.0

lip_pred = lip_model.predict(mouth_gray)

lip_class = np.argmax(lip_pred)

print("Lip class:", lip_class)

In [ ]:
from tensorflow.keras.models import Model

# Ensure base models are built
# model.build((None,48,48,1)) # Assuming 'model' is already built from previous executions
# lip_model.build((None,64,64,1)) # Assuming 'lip_model' is already built from previous executions

emotion_feature_model = Model(
    inputs=model.inputs,
    outputs=model.layers[-2].output
)

lip_feature_model = Model(
    inputs=lip_model.inputs,
    outputs=lip_model.layers[-2].output
)

emotion_features = emotion_feature_model.predict(face_gray)
lip_features = lip_feature_model.predict(mouth_gray)

combined_features = np.concatenate(
    (emotion_features, lip_features),
    axis=1
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Define the fusion model (if not already defined)
fusion_model = Sequential([
    Dense(128, activation='relu', input_shape=(384,)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

fusion_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

fusion_pred = fusion_model.predict(combined_features)

final_class = np.argmax(fusion_pred)

print("Final multimodal prediction:", final_class)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models

emotion_labels = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]

# Redefine the emotion detection model within this cell to ensure it's available
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(48,48,1)),
    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(7,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Note: This model has not been trained in this cell.
# For actual emotion prediction, ensure the model is loaded with trained weights
# or trained in a preceding cell.

img = cv2.imread(list(uploaded.keys())[0])
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

faces = face_cascade.detectMultiScale(img,1.3,5)

for (x,y,w,h) in faces:

    # face rectangle
    cv2.rectangle(img_rgb,(x,y),(x+w,y+h),(0,255,0),2)

    # lip area
    lip_x = x + int(w*0.2)
    lip_y = y + int(h*0.65)

    lip_w = int(w*0.6)
    lip_h = int(h*0.25)

    # lip rectangle
    cv2.rectangle(
        img_rgb,
        (lip_x,lip_y),
        (lip_x+lip_w,lip_y+lip_h),
        (0,255,0),
        2
    )

    # emotion detection
    face = img[y:y+h, x:x+w]

    face_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
    face_gray = cv2.resize(face_gray,(48,48))
    face_gray = face_gray.reshape(1,48,48,1)/255

    emotion_pred = model.predict(face_gray)
    emotion_class = np.argmax(emotion_pred)

    emotion_text = emotion_labels[emotion_class]

    # text on image
    cv2.putText(
        img_rgb,
        emotion_text,
        (x,y-10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (0,255,0),
        2
    )

plt.figure(figsize=(6,6))
plt.imshow(img_rgb)
plt.axis("off")

print("Detected Emotion:", emotion_text)

speech Recognition

In [ ]:
!pip install -U transformers librosa
!pip install librosa joblib

In [ ]:
import librosa
import numpy as np

audio, sr = librosa.load(audio_file, sr=22050)

mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
mfcc = np.mean(mfcc.T, axis=0)
mfcc = mfcc.reshape(1, -1)

In [ ]:
import librosa
audio_path = "/content/drive/MyDrive/audio_2026-03-15_10-18-15.wav"
audio, sr = librosa.load(audio_path, sr=16000)


In [ ]:
import librosa

audio_path = "/content/drive/MyDrive/audio_2026-03-15_10-18-15.wav"

audio, sr = librosa.load(audio_path, sr=16000)

In [ ]:
text = result["text"].lower()

if "happy" in text:
    voice_emotion = "Happy"
elif "sad" in text:
    voice_emotion = "Sad"
elif "angry" in text:
    voice_emotion = "Angry"
elif "fear" in text:
    voice_emotion = "Fear"
elif "surprise" in text:
    voice_emotion = "Surprise"
else:
    voice_emotion = "Neutral"

print("Voice Emotion:", voice_emotion)

In [ ]:
from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base"
)

result = asr({"array": audio, "sampling_rate": sr})

print("Text:", result["text"])

**live** **record**


**MFCC** Fearure **extraction**

In [ ]:
import librosa
import numpy as np

audio, sr = librosa.load(audio_file, sr=22050)

mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
mfcc = np.mean(mfcc.T, axis=0)
mfcc = mfcc.reshape(1, -1)

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js
from base64 import b64decode

def record(seconds=5, filename="speech.wav"):

    display(Javascript("""
    async function recordAudio(sec){
      const stream = await navigator.mediaDevices.getUserMedia({audio:true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);

      recorder.start();
      await new Promise(resolve => setTimeout(resolve, sec*1000));
      recorder.stop();

      await new Promise(resolve => recorder.onstop = resolve);

      const blob = new Blob(chunks);
      const reader = new FileReader();
      reader.readAsDataURL(blob);

      return new Promise(resolve=>{
        reader.onloadend = ()=>resolve(reader.result);
      });
    }
    """))

    audio = eval_js(f"recordAudio({seconds})")
    audio_bytes = b64decode(audio.split(',')[1])

    with open(filename, "wb") as f:
        f.write(audio_bytes)

    return filename

In [ ]:
audio_file = record(5)

print("Audio saved:", audio_file)

In [ ]:
import librosa
from transformers import pipeline

audio, sr = librosa.load(audio_file, sr=16000)

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base"
)

result = asr({"array": audio, "sampling_rate": sr})

print("Text:", result["text"])